### **Connect Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### **Import Libraries**

In [ ]:
import pandas as pd
import numpy  as np

### **Configuration**

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  KONFIGURASI UTAMA  —  sesuaikan path jika letak file berbeda
# ══════════════════════════════════════════════════════════════════════════════

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── Path Google Drive ──────────────────────────────────────────────────────────
BASE_PATH  = '/content/drive/MyDrive/Colab Datasets'

PATH_ID    = f'{BASE_PATH}/dataset_indonesia_final.csv'
PATH_EN    = f'{BASE_PATH}/dataset_english_final.csv'
PATH_MIXED = f'{BASE_PATH}/dataset_mixed_final.csv'

PATH_FINAL = f'{BASE_PATH}/combined_dataset_final.csv'
PATH_AUDIT = f'{BASE_PATH}/combined_dataset_final_audit.csv'

# ── Target ─────────────────────────────────────────────────────────────────────
TARGET_TOTAL     = 40_000
TARGET_PER_LABEL = TARGET_TOTAL // 2   # 20.000 hate / 20.000 non-hate

# ── Schema kolom ───────────────────────────────────────────────────────────────
EXPECTED_COLS = ['id', 'text', 'label', 'language', 'has_slang', 'has_abbrev', 'source']
OUTPUT_COLS   = ['id', 'text', 'label', 'language', 'has_slang', 'has_abbrev', 'source']

# Audit CSV: 9 kolom  (tambahan: original_id, source_file sebagai composite foreign key)
#   • original_id : ID baris di file sumber (sebelum merge & shuffle)
#   • source_file : nama file CSV asal baris tersebut
#   • Kombinasi (source_file + original_id) UNIK secara global → composite foreign key
AUDIT_COLS = [
    'id', 'original_id', 'source_file',
    'text', 'label', 'language', 'has_slang', 'has_abbrev', 'source',
]

# ── Separator visual ───────────────────────────────────────────────────────────
SEP  = '─' * 64
SEP2 = '═' * 64

print('✓ Konfigurasi selesai.')
print(f'  TARGET_TOTAL     : {TARGET_TOTAL:,}')
print(f'  TARGET_PER_LABEL : {TARGET_PER_LABEL:,}  (hate = 20.000 | non-hate = 20.000)')
print(f'  SEED             : {SEED}')

✓ Konfigurasi selesai.
  TARGET_TOTAL     : 40,000
  TARGET_PER_LABEL : 20,000  (hate = 20.000 | non-hate = 20.000)
  SEED             : 42


### **Helper Function**

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  FUNGSI BANTU
# ══════════════════════════════════════════════════════════════════════════════


def load_and_validate(path: str, lang_tag: str) -> pd.DataFrame:
    """
    Load CSV, validasi schema & tipe data, cetak ringkasan singkat.

    - Kolom ekstra di-drop otomatis (dengan notifikasi).
    - Kolom yang hilang  → raise ValueError.
    - Label non-biner   → raise ValueError.
    """
    print(f'  [{lang_tag.upper()}] Loading: {path}')
    df = pd.read_csv(path)

    missing_cols = [c for c in EXPECTED_COLS if c not in df.columns]
    extra_cols   = [c for c in df.columns    if c not in EXPECTED_COLS]

    if missing_cols:
        raise ValueError(f'[{lang_tag.upper()}] Kolom hilang: {missing_cols}')
    if extra_cols:
        print(f'  [{lang_tag.upper()}] ⚠ Kolom ekstra di-drop: {extra_cols}')

    df = df[EXPECTED_COLS].copy()

    # Normalisasi tipe data
    df['label']      = df['label'].astype(int)
    df['has_slang']  = df['has_slang'].astype(int)
    df['has_abbrev'] = df['has_abbrev'].astype(int)
    df['text']       = df['text'].astype(str).str.strip()
    df['language']   = df['language'].astype(str).str.lower().str.strip()
    df['source']     = df['source'].astype(str).str.strip()

    bad_labels = df[~df['label'].isin([0, 1])]
    if not bad_labels.empty:
        raise ValueError(
            f'[{lang_tag.upper()}] Ditemukan {len(bad_labels)} baris dengan label non-biner!'
        )

    n_hate    = int((df['label'] == 1).sum())
    n_nonhate = int((df['label'] == 0).sum())

    print(f'  [{lang_tag.upper()}] Baris    : {len(df):,}')
    print(f'  [{lang_tag.upper()}] Label    : hate={n_hate:,}  '
          f'non-hate={n_nonhate:,}  '
          f'(rasio hate={n_hate / len(df) * 100:.1f}%)')
    print(f'  [{lang_tag.upper()}] language : {sorted(df["language"].unique().tolist())}')
    print(f'  [{lang_tag.upper()}] source   : {sorted(df["source"].unique().tolist())}')
    return df


def sample_mixed_to_fill_gap(
    df_mixed         : pd.DataFrame,
    n_hate_needed    : int,
    n_nonhate_needed : int,
    seed             : int,
) -> tuple:
    """
    Ambil baris dari df_mixed untuk mengisi gap label global menuju 50:50.

    Strategi:
    - Prioritas utama : ambil TEPAT n_hate_needed (label=1) dan
                        n_nonhate_needed (label=0) → global persis 50:50.
    - Fallback        : jika stok salah satu label kurang, ambil semua yang
                        tersedia untuk label itu, lalu kompensasi dari label
                        lainnya agar total baris tetap = n_mixed_needed.
                        Global akan mendekati (tidak persis) 50:50.

    Returns
    -------
    df_sampled  : DataFrame hasil sampling
    exact_50_50 : bool — True jika berhasil persis 50:50
    """
    df0          = df_mixed[df_mixed['label'] == 0]
    df1          = df_mixed[df_mixed['label'] == 1]
    avail0       = len(df0)
    avail1       = len(df1)
    n_total_need = n_hate_needed + n_nonhate_needed

    shortage0 = max(0, n_nonhate_needed - avail0)
    shortage1 = max(0, n_hate_needed    - avail1)

    if shortage0 == 0 and shortage1 == 0:
        # ── Kasus ideal: stok cukup untuk 50:50 eksak ────────────────────────
        actual0     = n_nonhate_needed
        actual1     = n_hate_needed
        exact_50_50 = True

    else:
        # ── Fallback: ambil semua label yang kurang, kompensasi dari label lain ─
        exact_50_50 = False
        if shortage0 > 0 and shortage1 == 0:
            # non-hate (label=0) kurang
            actual0 = avail0
            actual1 = min(avail1, n_total_need - actual0)
        elif shortage1 > 0 and shortage0 == 0:
            # hate (label=1) kurang
            actual1 = avail1
            actual0 = min(avail0, n_total_need - actual1)
        else:
            # kedua label kurang — ambil semua yang tersedia
            actual0 = avail0
            actual1 = avail1

        print(f'  ⚠ WARNING: Stok Mixed tidak cukup untuk distribusi 50:50 yang persis.')
        print(f'       label=0 (non-hate) → dibutuhkan: {n_nonhate_needed:,} | tersedia: {avail0:,} | diambil: {actual0:,}')
        print(f'       label=1 (hate)     → dibutuhkan: {n_hate_needed:,} | tersedia: {avail1:,} | diambil: {actual1:,}')
        print(f'       Distribusi global akan MENDEKATI 50:50 semaksimal mungkin.')

    sampled0 = df0.sample(n=actual0, random_state=seed)
    sampled1 = df1.sample(n=actual1, random_state=seed)

    # Gabungkan dan kembalikan ke urutan indeks asli Mixed
    # (pengacakan global dilakukan di langkah [5] bersama semua bahasa)
    df_sampled = pd.concat([sampled0, sampled1]).sort_index()
    return df_sampled, exact_50_50


def prepare_for_merge(df: pd.DataFrame, source_file: str) -> pd.DataFrame:
    """
    Rename kolom 'id' → 'original_id' dan sisipkan kolom 'source_file'.
    Kombinasi (source_file, original_id) = composite foreign key yang unik global.
    """
    out = df.copy().rename(columns={'id': 'original_id'})
    out.insert(out.columns.get_loc('original_id') + 1, 'source_file', source_file)
    return out


def print_dist_table(df: pd.DataFrame, title: str = '') -> None:
    """Cetak tabel distribusi label per bahasa, lengkap dengan fitur informal."""
    if title:
        print(f'\n{SEP}\n  {title}\n{SEP}')

    total  = len(df)
    header = (
        f"  {'LANG':6s}  {'N':>7s}  {'%TOTAL':>7s}  "
        f"{'NON-HATE':>9s}  {'HATE':>6s}  {'%HATE':>6s}  "
        f"{'%SLANG':>7s}  {'%ABBREV':>8s}"
    )
    divider = (
        f"  {'-'*6}  {'-'*7}  {'-'*7}  {'-'*9}  {'-'*6}  "
        f"{'-'*6}  {'-'*7}  {'-'*8}"
    )
    print(header)
    print(divider)

    for lang in sorted(df['language'].unique()):
        sub = df[df['language'] == lang]
        n   = len(sub)
        pct = n / total * 100
        nh  = int((sub['label'] == 0).sum())
        h   = int((sub['label'] == 1).sum())
        ph  = h / n * 100
        ps  = sub['has_slang'].mean()  * 100
        pa  = sub['has_abbrev'].mean() * 100
        print(
            f'  {lang:6s}  {n:>7,}  {pct:>6.1f}%  '
            f'{nh:>9,}  {h:>6,}  {ph:>5.1f}%  {ps:>6.1f}%  {pa:>7.1f}%'
        )

    print(divider)
    nh_t = int((df['label'] == 0).sum())
    h_t  = int((df['label'] == 1).sum())
    ph_t = h_t / total * 100
    ps_t = df['has_slang'].mean()  * 100
    pa_t = df['has_abbrev'].mean() * 100
    print(
        f"  {'TOTAL':6s}  {total:>7,}  {'100.0%':>7s}  "
        f"{nh_t:>9,}  {h_t:>6,}  {ph_t:>5.1f}%  {ps_t:>6.1f}%  {pa_t:>7.1f}%"
    )
    print(SEP)


print('✓ Semua fungsi bantu telah didefinisikan.')

✓ Semua fungsi bantu telah didefinisikan.


### **[1] Load & Validate Dataset**

In [ ]:
print(SEP2)
print('  [1]  LOAD & VALIDASI DATASET')
print(SEP2)

df_id    = load_and_validate(PATH_ID,    'id')
print()
df_en    = load_and_validate(PATH_EN,    'en')
print()
df_mixed = load_and_validate(PATH_MIXED, 'mixed')

# ── Hitung kuota masing-masing sumber ─────────────────────────────────────────
n_id        = len(df_id)
n_en        = len(df_en)
n_mixed_raw = len(df_mixed)

# Jumlah baris Mixed yang dibutuhkan = sisa kuota setelah ID + EN diambil semua
n_mixed_needed = TARGET_TOTAL - n_id - n_en

print(f'\n{SEP}')
print(f'  Kalkulasi kuota per sumber:')
print(f'  TARGET_TOTAL               : {TARGET_TOTAL:,}')
print(f'  n_id   (semua ID)          : {n_id:,}')
print(f'  n_en   (semua EN)          : {n_en:,}')
print(f'  n_mixed tersedia (raw)     : {n_mixed_raw:,}')
print(f'  n_mixed DIBUTUHKAN         : {TARGET_TOTAL:,} − {n_id:,} − {n_en:,} = {n_mixed_needed:,}')
print(f'  n_mixed kelebihan (buffer) : {n_mixed_raw - n_mixed_needed:,}')
print(SEP)

# ── Guard: kuota harus positif dan tidak melebihi stok Mixed ──────────────────
if n_mixed_needed <= 0:
    raise ValueError(
        f'[ERROR] ID+EN ({n_id + n_en:,}) sudah ≥ TARGET_TOTAL ({TARGET_TOTAL:,})!'
    )
if n_mixed_needed > n_mixed_raw:
    raise ValueError(
        f'[ERROR] Stok Mixed {n_mixed_raw:,} < kebutuhan {n_mixed_needed:,}.'
    )

print(f'  ✓ Kuota Mixed = {n_mixed_needed:,} baris')

════════════════════════════════════════════════════════════════
  [1]  LOAD & VALIDASI DATASET
════════════════════════════════════════════════════════════════
  [ID] Loading: /content/drive/MyDrive/Colab Datasets/dataset_indonesia_final.csv
  [ID] Baris    : 11,794
  [ID] Label    : hate=5,794  non-hate=6,000  (rasio hate=49.1%)
  [ID] language : ['id']
  [ID] source   : ['tonneau_indonesian']

  [EN] Loading: /content/drive/MyDrive/Colab Datasets/dataset_english_final.csv
  [EN] Baris    : 12,000
  [EN] Label    : hate=6,000  non-hate=6,000  (rasio hate=50.0%)
  [EN] language : ['en']
  [EN] source   : ['tonneau_english']

  [MIXED] Loading: /content/drive/MyDrive/Colab Datasets/dataset_mixed_final.csv
  [MIXED] Baris    : 17,173
  [MIXED] Label    : hate=8,786  non-hate=8,387  (rasio hate=51.2%)
  [MIXED] language : ['mixed']
  [MIXED] source   : ['generated']

────────────────────────────────────────────────────────────────
  Kalkulasi kuota per sumber:
  TARGET_TOTAL             

## **[2] Count Gap Label (Target Global 50 : 50)**

In [ ]:
print(SEP2)
print('  [2]  HITUNG GAP LABEL  (TARGET GLOBAL 50:50)')
print(SEP2)

# ── Jumlah hate / non-hate yang sudah ada dari ID + EN ────────────────────────
hate_id    = int((df_id['label'] == 1).sum())
nonhate_id = int((df_id['label'] == 0).sum())
hate_en    = int((df_en['label'] == 1).sum())
nonhate_en = int((df_en['label'] == 0).sum())

hate_id_en    = hate_id    + hate_en
nonhate_id_en = nonhate_id + nonhate_en

# ── Gap yang harus dipenuhi oleh Mixed ────────────────────────────────────────
n_hate_from_mixed    = TARGET_PER_LABEL - hate_id_en
n_nonhate_from_mixed = TARGET_PER_LABEL - nonhate_id_en

print(f'  Target global              : hate={TARGET_PER_LABEL:,}  non-hate={TARGET_PER_LABEL:,}')
print(f'  {SEP[:50]}')
print(f'  ID   : hate={hate_id:,}   non-hate={nonhate_id:,}')
print(f'  EN   : hate={hate_en:,}   non-hate={nonhate_en:,}')
print(f'  {SEP[:50]}')
print(f'  ID+EN sudah ada            : hate={hate_id_en:,}   non-hate={nonhate_id_en:,}')
print(f'  Kekurangan (gap) dari Mixed: hate={n_hate_from_mixed:,}   non-hate={n_nonhate_from_mixed:,}')
print(f'  Total diambil dari Mixed   : {n_hate_from_mixed + n_nonhate_from_mixed:,}')
print()

# ── Verifikasi matematis ───────────────────────────────────────────────────────
# (20k - hate_id_en) + (20k - nonhate_id_en)
#  = 40k - (hate_id_en + nonhate_id_en)
#  = 40k - (n_id + n_en) = n_mixed_needed  ✓
assert n_hate_from_mixed + n_nonhate_from_mixed == n_mixed_needed, (
    f'[ASSERT] Gap sum ({n_hate_from_mixed}+{n_nonhate_from_mixed}) '
    f'!= n_mixed_needed ({n_mixed_needed})'
)

# ── Guard: jika ID+EN saja sudah melebihi target salah satu label ─────────────
if n_hate_from_mixed < 0:
    raise ValueError(
        f'[ERROR] ID+EN sudah memiliki {hate_id_en:,} hate, '
        f'melebihi target {TARGET_PER_LABEL:,}. Tidak bisa mencapai 50:50.'
    )
if n_nonhate_from_mixed < 0:
    raise ValueError(
        f'[ERROR] ID+EN sudah memiliki {nonhate_id_en:,} non-hate, '
        f'melebihi target {TARGET_PER_LABEL:,}. Tidak bisa mencapai 50:50.'
    )

print(f'  ✓ Gap terhitung. Mixed akan diambil:')
print(f'       label=1 (hate)     : {n_hate_from_mixed:,}')
print(f'       label=0 (non-hate) : {n_nonhate_from_mixed:,}')
print(f'       Total              : {n_mixed_needed:,}  '
      f'(= {TARGET_TOTAL:,} − {n_id:,} − {n_en:,}  ✓)')

════════════════════════════════════════════════════════════════
  [2]  HITUNG GAP LABEL  (TARGET GLOBAL 50:50)
════════════════════════════════════════════════════════════════
  Target global              : hate=20,000  non-hate=20,000
  ──────────────────────────────────────────────────
  ID   : hate=5,794   non-hate=6,000
  EN   : hate=6,000   non-hate=6,000
  ──────────────────────────────────────────────────
  ID+EN sudah ada            : hate=11,794   non-hate=12,000
  Kekurangan (gap) dari Mixed: hate=8,206   non-hate=8,000
  Total diambil dari Mixed   : 16,206

  ✓ Gap terhitung. Mixed akan diambil:
       label=1 (hate)     : 8,206
       label=0 (non-hate) : 8,000
       Total              : 16,206  (= 40,000 − 11,794 − 12,000  ✓)


## [3] **Sampling Code-Mixed**

In [ ]:
print(SEP2)
print('  [3]  SAMPLING CODE-MIXED')
print(SEP2)

avail_mixed_0 = int((df_mixed['label'] == 0).sum())
avail_mixed_1 = int((df_mixed['label'] == 1).sum())

print(f'  Stok Mixed tersedia:')
print(f'       label=0 (non-hate) : {avail_mixed_0:,}')
print(f'       label=1 (hate)     : {avail_mixed_1:,}')
print(f'  Akan diambil (target):')
print(f'       label=0 (non-hate) : {n_nonhate_from_mixed:,}')
print(f'       label=1 (hate)     : {n_hate_from_mixed:,}')
print()

df_mixed_sampled, is_exact_50_50 = sample_mixed_to_fill_gap(
    df_mixed,
    n_hate_needed    = n_hate_from_mixed,
    n_nonhate_needed = n_nonhate_from_mixed,
    seed             = SEED,
)

vc = df_mixed_sampled['label'].value_counts().sort_index().to_dict()
print(f'\n  Hasil sampling     : {len(df_mixed_sampled):,} baris')
print(f'  Distribusi label   : non-hate={vc.get(0, 0):,}  hate={vc.get(1, 0):,}')
print(f'  Distribusi 50:50   : {"✓ Persis" if is_exact_50_50 else "⚠ Mendekati (fallback aktif)"}')

# Jumlah Mixed yang benar-benar diambil (bisa < n_mixed_needed jika kedua label kurang)
n_mixed_taken  = len(df_mixed_sampled)
TARGET_TOTAL_ACTUAL = n_id + n_en + n_mixed_taken

print(f'  Total dataset final: {TARGET_TOTAL_ACTUAL:,}')
if TARGET_TOTAL_ACTUAL != TARGET_TOTAL:
    print(f'  ⚠ Total bergeser dari target {TARGET_TOTAL:,} karena stok Mixed tidak cukup.')
else:
    print(f'  ✓ Tepat {TARGET_TOTAL:,} baris Mixed terpilih')

════════════════════════════════════════════════════════════════
  [3]  SAMPLING CODE-MIXED
════════════════════════════════════════════════════════════════
  Stok Mixed tersedia:
       label=0 (non-hate) : 8,387
       label=1 (hate)     : 8,786
  Akan diambil (target):
       label=0 (non-hate) : 8,000
       label=1 (hate)     : 8,206


  Hasil sampling     : 16,206 baris
  Distribusi label   : non-hate=8,000  hate=8,206
  Distribusi 50:50   : ✓ Persis
  Total dataset final: 40,000
  ✓ Tepat 40,000 baris Mixed terpilih


### **[4] Concatenate**

In [ ]:
print(SEP2)
print('  [4]  PENGGABUNGAN (CONCATENATE)')
print(SEP2)

# Tambahkan kolom foreign-key ke masing-masing dataset
df_id_w    = prepare_for_merge(df_id,            'dataset_indonesia_final.csv')
df_en_w    = prepare_for_merge(df_en,            'dataset_english_final.csv')
df_mixed_w = prepare_for_merge(df_mixed_sampled, 'dataset_mixed_final.csv')

# Gabungkan: ID → EN → MIXED (urutan ini akan diacak di langkah [5])
df_merged = pd.concat([df_id_w, df_en_w, df_mixed_w], ignore_index=True)

print(f'  ID    : {n_id:,} baris   (100% dari dataset_indonesia_final.csv)')
print(f'  EN    : {n_en:,} baris   (100% dari dataset_english_final.csv)')
print(f'  MIXED : {n_mixed_taken:,} baris  '
      f'({n_mixed_taken / n_mixed_raw * 100:.1f}% dari dataset_mixed_final.csv)')
print(f'  {SEP}')
print(f'  Total gabungan (pre-shuffle) : {len(df_merged):,}')

# Verifikasi total
assert len(df_merged) == TARGET_TOTAL_ACTUAL, (
    f'[ASSERT] Total {len(df_merged):,} != TARGET_TOTAL_ACTUAL {TARGET_TOTAL_ACTUAL:,}'
)
print(f'  ✓ Total = {TARGET_TOTAL_ACTUAL:,}')

# ── Cek duplikat teks exact-match lintas dataset ───────────────────────────────
dup_all = int(df_merged.duplicated(subset=['text']).sum())
print(f'\n  Duplikat teks (exact match)  : {dup_all}')
if dup_all > 0:
    cross = int(
        df_merged[df_merged.duplicated(subset=['text'], keep=False)]
        .groupby('text')['language'].nunique().gt(1).sum()
    )
    flag = '⚠ WARNING — catat sebagai limitasi di Bab 3' if cross > 0 else '✓ Hanya dalam satu bahasa'
    print(f'  Duplikat lintas bahasa       : {cross}  [{flag}]')
else:
    print(f'  ✓ Tidak ada duplikat teks exact-match.')

# Tampilkan distribusi sebelum shuffle
print_dist_table(
    df_merged,
    'Distribusi PRE-SHUFFLE  (urutan: ID → EN → MIXED, belum diacak)'
)

════════════════════════════════════════════════════════════════
  [4]  PENGGABUNGAN (CONCATENATE)
════════════════════════════════════════════════════════════════
  ID    : 11,794 baris   (100% dari dataset_indonesia_final.csv)
  EN    : 12,000 baris   (100% dari dataset_english_final.csv)
  MIXED : 16,206 baris  (94.4% dari dataset_mixed_final.csv)
  ────────────────────────────────────────────────────────────────
  Total gabungan (pre-shuffle) : 40,000
  ✓ Total = 40,000

  Duplikat teks (exact match)  : 0
  ✓ Tidak ada duplikat teks exact-match.

────────────────────────────────────────────────────────────────
  Distribusi PRE-SHUFFLE  (urutan: ID → EN → MIXED, belum diacak)
────────────────────────────────────────────────────────────────
  LANG          N   %TOTAL   NON-HATE    HATE   %HATE   %SLANG   %ABBREV
  ------  -------  -------  ---------  ------  ------  -------  --------
  en       12,000    30.0%      6,000   6,000   50.0%     3.7%     12.9%
  id       11,794    29.5%  

### **[5] Shuffle Each Row & Reassign ID Global**

In [ ]:
print(SEP2)
print('  [5]  SHUFFLE PER-BARIS & REASSIGN ID GLOBAL')
print(SEP2)
print(f'  Seed : {SEED}')

# Snapshot 10 baris pertama SEBELUM shuffle (masih terurut per bahasa)
print(f'\n  10 baris pertama SEBELUM shuffle (masih terurut: ID → EN → MIXED):')
print(
    df_merged[['original_id', 'source_file', 'language', 'label']]
    .head(10).to_string(index=False)
)

# ── Acak posisi baris secara keseluruhan (row-level shuffle) ──────────────────
# .sample(frac=1)       = permutasi acak seluruh baris (row-level, integritas baris terjaga)
# .reset_index(drop=True) = hapus indeks lama setelah pengacakan
df_shuffled = df_merged.sample(frac=1, random_state=SEED).reset_index(drop=True)

# ── Assign ID global baru: 1 hingga TARGET_TOTAL_ACTUAL ───────────────────────
# insert di posisi 0 agar kolom 'id' tetap menjadi kolom pertama
df_shuffled.insert(0, 'id', range(1, len(df_shuffled) + 1))

# Snapshot 10 baris pertama SETELAH shuffle
print(f'\n  10 baris pertama SETELAH shuffle (bahasa sudah tercampur acak):')
print(
    df_shuffled[['id', 'original_id', 'source_file', 'language', 'label']]
    .head(10).to_string(index=False)
)

# ── Verifikasi ID ──────────────────────────────────────────────────────────────
assert df_shuffled['id'].nunique()  == len(df_shuffled),  '[ASSERT] ID tidak unik!'
assert int(df_shuffled['id'].min()) == 1,                 '[ASSERT] ID tidak mulai dari 1!'
assert int(df_shuffled['id'].max()) == len(df_shuffled),  '[ASSERT] ID tidak kontinu!'

print(f'\n  ✓ ID range   : 1 – {df_shuffled["id"].max():,}  (unik, kontinu, tanpa gap)')

════════════════════════════════════════════════════════════════
  [5]  SHUFFLE PER-BARIS & REASSIGN ID GLOBAL
════════════════════════════════════════════════════════════════
  Seed : 42

  10 baris pertama SEBELUM shuffle (masih terurut: ID → EN → MIXED):
 original_id                 source_file language  label
           1 dataset_indonesia_final.csv       id      1
           2 dataset_indonesia_final.csv       id      0
           3 dataset_indonesia_final.csv       id      1
           4 dataset_indonesia_final.csv       id      1
           5 dataset_indonesia_final.csv       id      0
           6 dataset_indonesia_final.csv       id      0
           7 dataset_indonesia_final.csv       id      0
           8 dataset_indonesia_final.csv       id      1
           9 dataset_indonesia_final.csv       id      0
          10 dataset_indonesia_final.csv       id      1

  10 baris pertama SETELAH shuffle (bahasa sudah tercampur acak):
 id  original_id                 source_file lan

### **[6] Verification**

In [ ]:
print(SEP2)
print('  [6]  VERIFIKASI DISTRIBUSI FINAL')
print(SEP2)

df_check = df_shuffled[['id'] + [c for c in OUTPUT_COLS if c != 'id']].copy()

# ── [6a] Tabel distribusi lengkap ─────────────────────────────────────────────
print_dist_table(df_check, '[6a] Distribusi FINAL (post-shuffle)')

# ── [6b] Keseimbangan label global ────────────────────────────────────────────
n_total   = len(df_check)
n_hate    = int((df_check['label'] == 1).sum())
n_nonhate = int((df_check['label'] == 0).sum())
imbalance = abs(n_hate - n_nonhate)

print(f'\n  [6b] Distribusi Label Global:')
print(f'       Hate     (label=1) : {n_hate:,}   ({n_hate / n_total * 100:.4f}%)')
print(f'       Non-hate (label=0) : {n_nonhate:,}   ({n_nonhate / n_total * 100:.4f}%)')
print(f'       Selisih            : {imbalance:,} baris')

if is_exact_50_50 and imbalance == 0:
    print(f'       ✓ LULUS — Persis 50:50  ({n_hate:,} hate : {n_nonhate:,} non-hate)')
elif imbalance == 0:
    print(f'       ✓ LULUS — Persis 50:50 (meskipun fallback aktif)')
else:
    pct_hate = n_hate / n_total * 100
    print(f'       ⚠ MENDEKATI 50:50 — Selisih {imbalance:,} baris '
          f'(rasio hate = {pct_hate:.2f}%)')
    print(f'       Catatan: fallback aktif karena stok Mixed tidak cukup untuk split eksak.')

# ── [6c] Komposisi bahasa ─────────────────────────────────────────────────────
print(f'\n  [6c] Komposisi Bahasa:')
for lang, expected, label in [
    ('id',    n_id,          'ID'),
    ('en',    n_en,          'EN'),
    ('mixed', n_mixed_taken, 'MIXED'),
]:
    n   = int((df_check['language'] == lang).sum())
    pct = n / n_total * 100
    ok  = '✓' if n == expected else f'⚠ expected {expected:,}'
    print(f'       {label:6s}: {n:,}  ({pct:.1f}%)  {ok}')

# ── [6d] NaN check ────────────────────────────────────────────────────────────
nan_counts = df_check.isnull().sum()
assert nan_counts.sum() == 0, (
    f'[ASSERT] Terdapat NaN: {nan_counts[nan_counts > 0].to_dict()}'
)
print(f'\n  [6d] NaN check          : ✓ LULUS  (0 nilai kosong)')

# ── [6e] Teks kosong ──────────────────────────────────────────────────────────
empty_texts = int((df_check['text'].str.strip() == '').sum())
assert empty_texts == 0, f'[ASSERT] Terdapat {empty_texts} teks kosong!'
print(f'  [6e] Teks kosong        : ✓ LULUS  ({empty_texts})')

# ── [6f] Label hanya 0 atau 1 ─────────────────────────────────────────────────
assert df_check[~df_check['label'].isin([0, 1])].empty
print(f'  [6f] Label valid (0/1)  : ✓ LULUS')

# ── [6g] Total baris ──────────────────────────────────────────────────────────
assert n_total == TARGET_TOTAL_ACTUAL, (
    f'[ASSERT] Total {n_total} != TARGET_TOTAL_ACTUAL {TARGET_TOTAL_ACTUAL}'
)
print(f'  [6g] Total baris        : ✓ LULUS  ({n_total:,})')

# ── [6h] ID unik dan kontinu ──────────────────────────────────────────────────
assert df_check['id'].nunique()  == n_total
assert int(df_check['id'].min()) == 1
assert int(df_check['id'].max()) == n_total
print(f'  [6h] ID range 1–{n_total:,}  : ✓ LULUS  (unik & kontinu)')

# ── [6i] Composite foreign key unik ───────────────────────────────────────────
fk_unique = df_shuffled.duplicated(subset=['source_file', 'original_id']).sum()
assert fk_unique == 0, f'[ASSERT] Composite FK tidak unik: {fk_unique} duplikat!'
print(f'  [6i] Composite FK unik  : ✓ LULUS  (source_file + original_id)')

print(f'\n  {SEP}')
print(f'  ✓ Semua verifikasi LULUS. Dataset final siap disimpan.')
print(f'  {SEP}')

════════════════════════════════════════════════════════════════
  [6]  VERIFIKASI DISTRIBUSI FINAL
════════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────────
  [6a] Distribusi FINAL (post-shuffle)
────────────────────────────────────────────────────────────────
  LANG          N   %TOTAL   NON-HATE    HATE   %HATE   %SLANG   %ABBREV
  ------  -------  -------  ---------  ------  ------  -------  --------
  en       12,000    30.0%      6,000   6,000   50.0%     3.7%     12.9%
  id       11,794    29.5%      6,000   5,794   49.1%    35.8%     52.4%
  mixed    16,206    40.5%      8,000   8,206   50.6%    49.0%     58.2%
  ------  -------  -------  ---------  ------  ------  -------  --------
  TOTAL    40,000   100.0%     20,000  20,000   50.0%    31.5%     42.9%
────────────────────────────────────────────────────────────────

  [6b] Distribusi Label Global:
       Hate     (label=1) : 20,000   (50.0000%)
       N

### **[7] Save**

In [ ]:
print(SEP2)
print('  [7]  SIMPAN DATASET FINAL')
print(SEP2)

# ── [7a] Final CSV — 7 kolom standar ──────────────────────────────────────────
df_final = df_shuffled[OUTPUT_COLS].copy()

# Verifikasi schema sebelum menyimpan
assert list(df_final.columns) == OUTPUT_COLS, (
    f'[ERROR] Kolom final tidak sesuai: {list(df_final.columns)}'
)

df_final.to_csv(PATH_FINAL, index=False)

print(f'  [7a] Final CSV  : {PATH_FINAL}')
print(f'       Kolom      : {list(df_final.columns)}')
print(f'       Baris      : {len(df_final):,}')

# ── [7b] Audit CSV — 9 kolom (+ original_id + source_file) ───────────────────
df_audit = df_shuffled[AUDIT_COLS].copy()

# Verifikasi schema audit sebelum menyimpan
assert list(df_audit.columns) == AUDIT_COLS, (
    f'[ERROR] Kolom audit tidak sesuai: {list(df_audit.columns)}'
)

df_audit.to_csv(PATH_AUDIT, index=False)

print(f'\n  [7b] Audit CSV  : {PATH_AUDIT}')
print(f'       Kolom      : {AUDIT_COLS}')
print(f'       Baris      : {len(df_audit):,}')

print(f'\n  ✓ Kedua file berhasil disimpan.')

════════════════════════════════════════════════════════════════
  [7]  SIMPAN DATASET FINAL
════════════════════════════════════════════════════════════════
  [7a] Final CSV  : /content/drive/MyDrive/Colab Datasets/combined_dataset_final.csv
       Kolom      : ['id', 'text', 'label', 'language', 'has_slang', 'has_abbrev', 'source']
       Baris      : 40,000

  [7b] Audit CSV  : /content/drive/MyDrive/Colab Datasets/combined_dataset_final_audit.csv
       Kolom      : ['id', 'original_id', 'source_file', 'text', 'label', 'language', 'has_slang', 'has_abbrev', 'source']
       Baris      : 40,000

  ✓ Kedua file berhasil disimpan.


## [8] **Conclusion**

In [ ]:
print('\n' + SEP2)
print('  RINGKASAN AKHIR — DATASET FINAL GABUNGAN')
print(SEP2)

df_f = df_final

print(f'  Total baris      : {len(df_f):,}')
print(f'  ID range         : {int(df_f["id"].min())} – {int(df_f["id"].max()):,}')
print(f'  Kolom output     : {list(df_f.columns)}')

# ── Komposisi Bahasa ───────────────────────────────────────────────────────────
print(f'\n  +─ Komposisi Bahasa ─────────────────────────────────────────────+')
for lang, label, n_src, take_all in [
    ('id',    'ID',    n_id,          True),
    ('en',    'EN',    n_en,          True),
    ('mixed', 'MIXED', n_mixed_taken, False),
]:
    sub  = df_f[df_f['language'] == lang]
    pct  = len(sub) / len(df_f) * 100
    flag = 'diambil semua' if take_all else f'diambil {n_src / n_mixed_raw * 100:.1f}% dari raw'
    print(f'  |  {label:6s}: {len(sub):>7,} baris  ({pct:.1f}%)   [{flag}]')
print(f'  +────────────────────────────────────────────────────────────────+')

# ── Distribusi Label Global ────────────────────────────────────────────────────
print(f'\n  +─ Distribusi Label (Global) ─────────────────────────────────────+')
for lbl, name in [(1, 'Hate'), (0, 'Non-Hate')]:
    n   = int((df_f['label'] == lbl).sum())
    pct = n / len(df_f) * 100
    print(f'  |  {name:10s} (label={lbl}) : {n:>7,}  ({pct:.2f}%)')
status_label = '50:50 EKSAK ✓' if is_exact_50_50 and abs(n_hate - n_nonhate) == 0 \
               else f'MENDEKATI 50:50 (selisih {abs(n_hate - n_nonhate):,} baris)'
print(f'  |  Status distribusi: {status_label}')
print(f'  +────────────────────────────────────────────────────────────────+')

# ── Label per Bahasa ───────────────────────────────────────────────────────────
print(f'\n  +─ Label per Bahasa ──────────────────────────────────────────────+')
for lang in ['id', 'en', 'mixed']:
    sub = df_f[df_f['language'] == lang]
    h   = int((sub['label'] == 1).sum())
    nh  = int((sub['label'] == 0).sum())
    print(f'  |  {lang.upper():6s}: hate={h:>7,}  non-hate={nh:>7,}  total={len(sub):>7,}')
print(f'  +────────────────────────────────────────────────────────────────+')

# ── Tabel Transisi Jumlah Data ─────────────────────────────────────────────────
print(f'\n  +─ Tabel Transisi Jumlah Data ────────────────────────────────────+')
print(f'  |  MIXED tersedia (raw)              : {n_mixed_raw:>7,}')
print(f'  |  MIXED setelah gap-fill sampling   : {n_mixed_taken:>7,}')
print(f'  |  Setelah concat  (ID + EN + MIXED) : {TARGET_TOTAL_ACTUAL:>7,}')
print(f'  |  Setelah shuffle & reassign ID     : {len(df_f):>7,}')
print(f'  +────────────────────────────────────────────────────────────────+')

# ── Audit File Info ────────────────────────────────────────────────────────────
print(f'\n  +─ Audit File Info ───────────────────────────────────────────────+')
print(f'  |  Kolom audit   : {AUDIT_COLS}')
print(f'  |  Composite FK  : (source_file, original_id)  — unik secara global')
print(f'  +────────────────────────────────────────────────────────────────+')

# ── Preview ────────────────────────────────────────────────────────────────────
print(f'\n  5 baris pertama (urutan acak):')
print(df_f[['id', 'language', 'label', 'has_slang', 'has_abbrev', 'source']].head(5).to_string(index=False))
print(f'\n  5 baris terakhir:')
print(df_f[['id', 'language', 'label', 'has_slang', 'has_abbrev', 'source']].tail(5).to_string(index=False))

print(f'\n{SEP2}')
print(f'  SELESAI. File tersimpan di:')
print(f'  → {PATH_FINAL}')
print(f'  → {PATH_AUDIT}')
print(SEP2)


════════════════════════════════════════════════════════════════
  RINGKASAN AKHIR — DATASET FINAL GABUNGAN
════════════════════════════════════════════════════════════════
  Total baris      : 40,000
  ID range         : 1 – 40,000
  Kolom output     : ['id', 'text', 'label', 'language', 'has_slang', 'has_abbrev', 'source']

  +─ Komposisi Bahasa ─────────────────────────────────────────────+
  |  ID    :  11,794 baris  (29.5%)   [diambil semua]
  |  EN    :  12,000 baris  (30.0%)   [diambil semua]
  |  MIXED :  16,206 baris  (40.5%)   [diambil 94.4% dari raw]
  +────────────────────────────────────────────────────────────────+

  +─ Distribusi Label (Global) ─────────────────────────────────────+
  |  Hate       (label=1) :  20,000  (50.00%)
  |  Non-Hate   (label=0) :  20,000  (50.00%)
  |  Status distribusi: 50:50 EKSAK ✓
  +────────────────────────────────────────────────────────────────+

  +─ Label per Bahasa ──────────────────────────────────────────────+
  |  ID    : hate=  5